In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# ===============================
# CONFIG
# ===============================
INPUT_FILE = "/kaggle/input/qa-data1/qa_data.json"
OUTPUT_FILE = "/kaggle/working/qa_data_summarized_flan_t5.json"

MODEL_NAME = "google/flan-t5-large"
MAX_INPUT_LEN = 1024
MAX_SUMMARY_LEN = 180

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ===============================
# LOAD MODEL
# ===============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# ===============================
# QUESTION-AWARE SUMMARIZATION
# ===============================
def summarize_knowledge_with_question(knowledge, question):
    prompt = (
        "Summarize the knowledge below so that it directly helps answer the question.\n\n"
        f"Question: {question}\n"
        f"Knowledge: {knowledge}"
    )

    inputs = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_INPUT_LEN,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=MAX_SUMMARY_LEN,
            num_beams=4
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# ===============================
# LOAD DATA (JSON / JSONL)
# ===============================
def load_json_or_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        first_char = f.read(1)
        f.seek(0)
        if first_char == "[":
            data = json.load(f)
        else:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
    return data

data = load_json_or_jsonl(INPUT_FILE)
print(f"Loaded {len(data)} samples")

# ===============================
# PROCESS + SAVE
# ===============================
new_data = []

for sample in tqdm(data, desc="Summarizing with FLAN-T5"):
    summarized_knowledge = summarize_knowledge_with_question(
        sample["knowledge"],
        sample["question"]
    )

    new_data.append({
        "knowledge": summarized_knowledge,
        "question": sample["question"],
        "right_answer": sample["right_answer"],
        "hallucinated_answer": sample["hallucinated_answer"]
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(new_data, f, indent=2, ensure_ascii=False)

print(f"\n✅ File saved at: {OUTPUT_FILE}")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded 10000 samples


Summarizing with FLAN-T5: 100%|██████████| 10000/10000 [4:00:19<00:00,  1.44s/it] 


✅ File saved at: /kaggle/working/qa_data_summarized_flan_t5.json
